In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
df = pd.read_csv("../data/raw/tcc_ceds_music.csv")

In [6]:
df.head()

,Unnamed: 0,artist_name,track_name,release_date,genre,lyrics,len,dating,violence,world/life,...,sadness,feelings,danceability,loudness,acousticness,instrumentalness,valence,energy,topic,age
0,0,mukesh,mohabbat bhi jhoothi,1950,pop,hold time feel break feel untrue convince spea...,95,0.000598,0.063746,0.000598,...,0.380299,0.117175,0.357739,0.454119,0.997992,0.901822,0.339448,0.137110,sadness,1.0
1,4,frankie laine,i believe,1950,pop,believe drop rain fall grow believe darkest ni...,51,0.035537,0.096777,0.443435,...,0.001284,0.001284,0.331745,0.647540,0.954819,0.000002,0.325021,0.263240,world/life,1.0
2,6,johnnie ray,cry,1950,pop,sweetheart send letter goodbye secret feel bet...,24,0.002770,0.002770,0.002770,...,0.002770,0.225422,0.456298,0.585288,0.840361,0.000000,0.351814,0.139112,music,1.0
3,10,pérez prado,patricia,1950,pop,kiss lips want stroll charm mambo chacha merin...,54,0.048249,0.001548,0.001548,...,0.225889,0.001548,0.686992,0.744404,0.083935,0.199393,0.775350,0.743736,romantic,1.0
4,12,giorgos papadopoulos,apopse eida oneiro,1950,pop,till darling till matter know till dream live ...,48,0.001350,0.001350,0.417772,...,0.068800,0.001350,0.291671,0.646489,0.975904,0.000246,0.597073,0.394375,romantic,1.0


In [7]:
df.columns

Index(['Unnamed: 0', 'artist_name', 'track_name', 'release_date', 'genre',
       'lyrics', 'len', 'dating', 'violence', 'world/life', 'night/time',
       'shake the audience', 'family/gospel', 'romantic', 'communication',
       'obscene', 'music', 'movement/places', 'light/visual perceptions',
       'family/spiritual', 'like/girls', 'sadness', 'feelings', 'danceability',
       'loudness', 'acousticness', 'instrumentalness', 'valence', 'energy',
       'topic', 'age'],
      dtype='object')

In [8]:
df = df[['track_name', 'artist_name', 'genre']]

In [9]:
df.head()

,track_name,artist_name,genre
0,mohabbat bhi jhoothi,mukesh,pop
1,i believe,frankie laine,pop
2,cry,johnnie ray,pop
3,patricia,pérez prado,pop
4,apopse eida oneiro,giorgos papadopoulos,pop


In [10]:
df.isnull().sum()

track_name     0
artist_name    0
genre          0
dtype: int64

In [11]:
df = df.fillna('')

In [13]:
df['combined_features'] = (
    df['genre'] + ' ' +
    df['artist_name'] + ' ' +
    df['track_name']
)

In [14]:
df[['combined_features']].head()

,combined_features
0,pop mukesh mohabbat bhi jhoothi
1,pop frankie laine i believe
2,pop johnnie ray cry
3,pop pérez prado patricia
4,pop giorgos papadopoulos apopse eida oneiro


In [15]:
tfidf = TfidfVectorizer(stop_words='english')

In [16]:
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

In [17]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [28]:
def recommend_songs(song_name, top_n=10):

    song_name = song_name.lower()

    df['track_name_lower'] = df['track_name'].str.lower()

    idx = df[df['track_name_lower'] == song_name].index

    if len(idx) == 0:
        return "Song not found."

    idx = idx[0]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    song_indices = [i[0] for i in sim_scores]

    return df[['track_name', 'artist_name', 'genre']].iloc[song_indices]

In [29]:
recommend_songs("cry")

,track_name,artist_name,genre
56,here am i,johnnie ray,pop
12887,keep on loving you,johnnie taylor,blues
15294,too many memories,johnnie taylor,blues
14542,there's nothing i wouldn't do,johnnie taylor,blues
14679,lately,johnnie taylor,blues
13887,steal away,johnnie taylor,blues
14701,don't make me late,johnnie taylor,blues
12881,"baby, we've got love",johnnie taylor,blues
13247,i got to love somebody's baby,johnnie taylor,blues
12882,i need lots of love,johnnie taylor,blues


In [30]:
recommend_songs("believer")

,track_name,artist_name,genre
24697,now you see it (now you don't),ozzy osbourne,rock
25365,no more tears,ozzy osbourne,rock
25386,road to nowhere,ozzy osbourne,rock
24722,so tired,ozzy osbourne,rock
24679,you're no different,ozzy osbourne,rock
26096,dreamer,ozzy osbourne,rock
24949,fool like you,ozzy osbourne,rock
25321,time after time,ozzy osbourne,rock
24476,little dolls,ozzy osbourne,rock
24983,shot in the dark,ozzy osbourne,rock
